# 02 — Training (native bf16, torch.compile, W&B, resume-safe)

Edit the **CONFIG** cell to pick the model and run. Checkpoints (`last.pth`/`best.pth`) and the
W&B run-id persist to `PERSIST_DIR`, so re-running after a session ends **resumes** automatically —
the key to spanning the three 6-hour GPU windows.

Suggested runs: Day 1 `resnet50` then `vit_b`; Day 2 `vit_l` (hero), plus `vit_l` with
`use_stain_aug=False` for the ablation, plus `phikon`.

In [ ]:
# Locate shared.ipynb regardless of the kernel's working directory, then run it.
from pathlib import Path as _P
_sh = next((p for p in [_P.cwd() / "shared.ipynb",
                        _P("/workspace/shared/ft004/shared.ipynb")] if p.exists()), None)
if _sh is None:
    _hits = list(_P.cwd().rglob("shared.ipynb")) or list(_P("/workspace").rglob("shared.ipynb"))
    _sh = _hits[0] if _hits else None
assert _sh, "shared.ipynb not found - keep it beside the notebooks or in /workspace/shared/ft004"
print("running", _sh)
get_ipython().run_line_magic("run", str(_sh))
import os
from pathlib import Path
PERSIST_DIR = Path(os.environ.get("PERSIST_DIR", "/workspace/shared/ft004"))
set_seed(42)


In [ ]:
# ===================== CONFIG =====================
cfg = dict(
    model="vit_l",            # resnet50 | vit_b | vit_l | swin_b | phikon
    use_stain_aug=True,       # set False for the ablation counterpart
    hed_theta=0.05,
    img_size=224,
    epochs=12,
    batch_size=256,           # ViT-L on MI300X can push 512; ResNet/ViT-B 256
    eval_batch_size=512,
    lr=1e-4,                  # head LR (backbone uses lr * backbone_lr_mult)
    backbone_lr_mult=0.1,
    weight_decay=0.05,
    warmup_epochs=1,
    label_smoothing=0.1,
    num_workers=8,            # needs container --ipc=host / --shm-size; lower if bus errors
    pin_memory=True,
    prefetch_factor=4,
    use_compile=True,         # torch.compile(reduce-overhead); falls back to eager on failure
    resume=True,
)
run_name = f"{cfg['model']}_{'stain' if cfg['use_stain_aug'] else 'nostain'}"
ckpt_dir = PERSIST_DIR / "checkpoints" / run_name
ckpt_dir.mkdir(parents=True, exist_ok=True)
device = get_device()
amp_dtype = get_amp_dtype(device)
print(f"run={run_name} device={device} amp={amp_dtype}")


In [ ]:
# Data
paths = ensure_dataset(PERSIST_DIR)
train_loader, val_loader, classes = make_dataloaders(
    paths["NCT-CRC-HE-100K"], paths["CRC-VAL-HE-7K"], cfg)
print(f"classes={classes}")
print(f"train batches={len(train_loader)} val batches={len(val_loader)}")


In [ ]:
# Model, optimizer, scheduler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

model = build_model(cfg, num_classes=len(classes)).to(device)
optimizer = torch.optim.AdamW(get_param_groups(model, cfg))
warmup = LinearLR(optimizer, start_factor=0.1, total_iters=max(1, cfg["warmup_epochs"]))
cosine = CosineAnnealingLR(optimizer, T_max=max(1, cfg["epochs"] - cfg["warmup_epochs"]),
                           eta_min=cfg["lr"] * 0.01)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[cfg["warmup_epochs"]])
criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

# fp16 (non-MI300X) needs a GradScaler; bf16 on MI300X does NOT.
scaler = torch.cuda.amp.GradScaler() if amp_dtype == torch.float16 else None

if cfg["use_compile"] and device.type == "cuda":
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("torch.compile: enabled (reduce-overhead)")
    except Exception as e:
        print(f"torch.compile failed -> eager ({e})")


In [ ]:
# Resume from last.pth if present (atomic checkpoints written every epoch)
import wandb
start_epoch, best_f1 = 0, 0.0
last_path = ckpt_dir / "last.pth"
if cfg["resume"] and last_path.exists():
    ck = load_checkpoint(last_path)
    raw_module(model).load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1
    best_f1 = ck.get("best_metric", 0.0)
    logger.info(f"Resumed {run_name} from epoch {start_epoch} (best_f1={best_f1:.2f})")

# Persist W&B run id so a restarted session continues the SAME run
run_id_file = ckpt_dir / "wandb_run_id.txt"
run_id = run_id_file.read_text().strip() if run_id_file.exists() else wandb.util.generate_id()
run_id_file.write_text(run_id)
wandb.init(project="finetuning_004", id=run_id, resume="allow", name=run_name, config=cfg)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, amp_dtype, scaler, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = device.type == "cuda"
    t0 = time.time()
    for step, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                out = model(x)
                loss = criterion(out.float(), y)
        else:
            out = model(x); loss = criterion(out, y)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * y.size(0); total += y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        if step % 50 == 0:
            ips = total / max(1e-6, time.time() - t0)
            logger.info(f"ep{epoch} step{step}/{len(loader)} loss {loss.item():.4f} "
                        f"acc {100*correct/total:.1f}% {ips:.0f} img/s")
    return total_loss / total, 100 * correct / total


In [ ]:
# Training loop — eval every epoch, log to W&B, save best + last (resume-safe)
for epoch in range(start_epoch, cfg["epochs"]):
    logger.info(f"===== epoch {epoch+1}/{cfg['epochs']} =====")
    try:
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion,
                                           device, amp_dtype, scaler, epoch)
        scheduler.step()
        preds, labels = run_inference(model, val_loader, device, amp_dtype)
        m = compute_metrics(labels, preds, classes)
        logger.info(f"val acc {m['accuracy']:.2f}% macro-F1 {m['macro_f1']:.2f}%")

        wandb.log({"epoch": epoch, "train/loss": tr_loss, "train/acc": tr_acc,
                   "val/acc": m["accuracy"], "val/macro_f1": m["macro_f1"],
                   "lr": optimizer.param_groups[0]["lr"],
                   **{f"val/f1_{c}": v for c, v in m["per_class_f1"].items()}})
        try:
            wandb.log({"val/confusion": wandb.plot.confusion_matrix(
                probs=None, y_true=labels.tolist(), preds=preds.tolist(),
                class_names=classes)})
        except Exception as e:
            logger.warning(f"wandb confusion log skipped ({e})")

        state = dict(epoch=epoch, model=raw_module(model).state_dict(),
                     optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                     best_metric=best_f1, metrics=m, cfg=cfg, classes=classes)
        save_checkpoint(state, last_path)
        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]; state["best_metric"] = best_f1
            save_checkpoint(state, ckpt_dir / "best.pth")
            logger.info(f"** new best macro-F1 {best_f1:.2f}% -> best.pth **")
    except Exception as e:
        logger.exception(f"epoch {epoch} failed; last.pth retained for resume ({e})")
        raise

logger.info(f"DONE {run_name}: best macro-F1 {best_f1:.2f}%")
wandb.finish()
